# 3. Thu thập references và độ phủ metadata

## 3.1 Truy vấn và ghép kết quả

S2 batch nhận danh sách paper ID; kết quả được gắn lại bằng `externalIds.ArXiv`, không chỉ
dựa vào vị trí trả về. Entry null, thiếu hoặc trùng định danh được kiểm tra riêng. Danh sách
references được đối chiếu với `referenceCount`; khi thiếu, crawler lấy thêm trang và vẫn
giữ trạng thái partial nếu chưa thu đủ. Batch quá lớn được chia nhỏ thay vì lặp toàn bộ.

Working data giữ các references đã thu được, gồm cả tài liệu không có arXiv ID. Course
export chỉ giữ cạnh có đích arXiv. Số dòng của hai lớp vì vậy có thể khác nhau mà không phải
merger làm mất cạnh hợp lệ theo schema công bố.

## 3.2 Ý nghĩa trạng thái

| Trạng thái | Ý nghĩa thống kê |
|---|---|
| success | S2 trả references đã đối chiếu số kỳ vọng |
| empty | S2 xác nhận referenceCount bằng 0 |
| partial / unavailable | Chưa có danh sách đầy đủ hoặc phản hồi bất thường |
| not_indexed | Không có record S2 tương ứng ở lần kiểm tra |
| bib_fallback | Có thông tin từ nguồn BibTeX, không mặc định là cạnh S2 |
| api_error | Truy vấn thất bại hoặc đang bị hoãn; không suy ra danh sách rỗng |

Cạnh trích dẫn và metadata của tài liệu được trích có provenance riêng. Việc bổ sung được
S2 ID cho một reference fallback không chứng minh S2 đã trả cạnh citing → reference đó.
Theo QA dòng 20–21, trường `ref_submission_date` được dùng `publicationDate` của S2 hoặc
ngày lấy qua arXiv ID. Pipeline ghi nguồn ngày, giữ ngày đã có và không suy ngày từ year.
Đây là quy ước của bảng references trong đồ án; nó không đổi nghĩa ngày submission của papers.

## 3.3 Retry và hạn mức

Crawler tôn trọng `Retry-After`, ghi cooldown để giữ qua restart và giới hạn số lần thử.
Khi S2 đang cooldown, phần chỉ còn lỗi references được hoãn để tránh ghi lặp cùng trạng
thái, trong khi nguồn chưa đủ vẫn cần được xử lý. Không biến API error thành thành công
để rút ngắn hàng đợi. Các batch dùng chung request được gắn request group để không tính
một POST thành nhiều HTTP request khi tổng hợp theo paper.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
data = json.loads((Path.cwd().parent / 'report_summary.json').read_text(encoding='utf-8'))
scope, m1, m2 = data['scope'], data['m1'], data['m2']
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False, 'axes.spines.right': False})
print(f"Snapshot: {scope['papers']} paper / {scope['versions']} version; báo cáo ngày {data['report_date']}")

In [ ]:
statuses = pd.Series(m1['reference_status'], name='papers'); display(statuses.to_frame())
statuses.plot.bar(title='Kết quả references theo paper', ylabel='Số paper'); plt.tight_layout(); plt.show()
n_verified = m1['verified_reference_papers']
display(pd.DataFrame([{'scope':'S2 success/empty', 'papers':n_verified, 'fraction':n_verified/scope['papers']},
                      {'scope':'API error/chưa đủ bằng chứng', 'papers':scope['papers']-n_verified,
                       'fraction':1-n_verified/scope['papers']}]))
display(pd.DataFrame([{'working_reference_rows':m1['working_reference_rows'],
                      'course_arxiv_edges':m1['arxiv_reference_rows'],
                      'mean_on_verified_papers':m1['mean_references_verified_papers']}]))

## 3.4 Công thức và mẫu số

Tỷ lệ paper có kết quả API đã xác minh là
$(N_{\mathrm{success}}+N_{\mathrm{empty}})/N_{\mathrm{paper}}$.
References trung bình chỉ dùng tập $Q$ có `success/empty`:
$\overline{R}=\sum_{p\in Q}R_p/|Q|$. Empty đã xác minh đóng góp 0; API error không đóng góp
0 vào trung bình. Trung bình có điều kiện này cũng không được suy ra cho toàn bộ nhóm.

Bản snapshot không tải toàn parent chain, nên có thể thiếu cạnh cũ không nằm trong các
lô được chọn. Các trường thiếu dưới đây mô tả các dòng thực sự có trong snapshot,
không mô tả toàn bộ references của các paper còn lỗi.

In [ ]:
missing = pd.Series(m1['references_missing_fraction'], name='missing_fraction')
display(missing.to_frame())
missing.plot.bar(ylim=(0,1), ylabel='Tỷ lệ thiếu', title='Metadata thiếu trên working references đã có')
plt.tight_layout(); plt.show()
display(Markdown(f"Có **{n_verified}/{scope['papers']} paper ({n_verified/scope['papers']:.2%})** có kết quả S2 đã xác minh. "
                 f"Trung bình có điều kiện là **{m1['mean_references_verified_papers']:.2f} references/paper**. "
                 'Không dùng số dòng hiện có chia cho mọi paper như một ước lượng đầy đủ.'))

## 3.5 Phân tích và bước hoàn thiện

Độ phủ references là điểm chưa hoàn tất đáng kể của snapshot. Cần truy vấn lại các paper
lỗi theo giới hạn dịch vụ, kiểm tra phân trang và lưu trạng thái cùng lần đo. Khi đủ nguồn,
đối chiếu bibliography giúp phát hiện thông tin thiếu, nhưng không thay thế tùy tiện yêu
cầu thu thập metadata từ S2. Báo cáo hiện giữ rõ sự khác nhau giữa nguồn đã audit tốt và
references đã thu đủ; không dùng số paper có TeX làm số paper crawling hoàn tất.

## Tài liệu và mã tham chiếu

- [Đặc tả M1](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/).
- [arXiv API: cấu trúc truy vấn và Atom](https://info.arxiv.org/help/api/user-manual.html).
- [Quy định truy cập arXiv](https://info.arxiv.org/help/api/tou.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã triển khai: [src/crawler/reference_scraper.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/6cd8dac6d1491083286d538dd7fffacf66f4cc4c/src/crawler/reference_scraper.py).
- Mã triển khai: [src/crawler/reference_batch.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/6cd8dac6d1491083286d538dd7fffacf66f4cc4c/src/crawler/reference_batch.py).
- Mã triển khai: [src/crawler/enrich_references.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/6cd8dac6d1491083286d538dd7fffacf66f4cc4c/src/crawler/enrich_references.py).
- Mã triển khai: [src/crawler/observations.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/6cd8dac6d1491083286d538dd7fffacf66f4cc4c/src/crawler/observations.py).